# RAG & Embedding Foundations

Companion notebook for **IM Session 31 - RAG & Embedding Foundations**.

We generate embeddings, compute similarity, build a full retrieval pipeline over a 20-article customer support knowledge base, and take a first look at storing embeddings in a vector database (Chroma) for fast lookup at scale.

In [1]:
import pandas as pd
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

model = SentenceTransformer("all-MiniLM-L6-v2")
kb = pd.read_csv("support_kb_articles.csv")
print(kb.shape)
kb.head()

/Users/suman/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/suman/Library/Python/3.9/lib/python/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


(20, 4)


,doc_id,title,category,content
0,KB001,Standard Delivery Timelines,delivery,Standard delivery for in-stock items takes 2 t...
1,KB002,Express Delivery Option,delivery,"Express delivery is available in Gurugram, Mum..."
2,KB003,Tracking a Delayed Order,delivery,If an order has not arrived within the estimat...
3,KB004,Delivery Address Changes,delivery,A delivery address can be changed only before ...
4,KB005,Return Window for Standard Items,returns,Most items can be returned within 7 days of de...


## 1. Text to vectors

In [2]:
sentence_a = "The delivery arrived two days late."
sentence_b = "My order showed up 48 hours behind schedule."
sentence_c = "The weather in Chennai was sunny today."

embedding_a = model.encode([sentence_a])
embedding_b = model.encode([sentence_b])
embedding_c = model.encode([sentence_c])

print(f"Embedding length: {embedding_a.shape[1]}")
print(f"Similar meaning:   {cosine_similarity(embedding_a, embedding_b)[0][0]:.3f}")
print(f"Unrelated meaning: {cosine_similarity(embedding_a, embedding_c)[0][0]:.3f}")

Embedding length: 384
Similar meaning:   0.575
Unrelated meaning: 0.103


## 2. TF-IDF vs. a real embedding model

A quick, concrete demonstration of why bag-of-words (TF-IDF) is not the same as an embedding.

In [3]:
from sklearn.feature_extraction.text import TfidfVectorizer

paraphrase_query = "How long do I have to send something back?"

tfidf = TfidfVectorizer(stop_words="english")
tfidf_doc_vectors = tfidf.fit_transform(kb["content"])
tfidf_query_vector = tfidf.transform([paraphrase_query])
tfidf_scores = cosine_similarity(tfidf_query_vector, tfidf_doc_vectors)[0]

kb_tfidf = kb.copy()
kb_tfidf["tfidf_similarity"] = tfidf_scores
print("TF-IDF top match:")
print(kb_tfidf.sort_values("tfidf_similarity", ascending=False)[["doc_id", "tfidf_similarity"]].head(1))

TF-IDF top match:
  doc_id  tfidf_similarity
0  KB001               0.0


In [4]:
doc_embeddings = model.encode(kb["content"].tolist())
query_embedding = model.encode([paraphrase_query])
real_scores = cosine_similarity(query_embedding, doc_embeddings)[0]

kb_real = kb.copy()
kb_real["embedding_similarity"] = real_scores
print("Real embedding model top match:")
print(kb_real.sort_values("embedding_similarity", ascending=False)[["doc_id", "embedding_similarity"]].head(1))

Real embedding model top match:
  doc_id  embedding_similarity
4  KB005              0.651613


/Users/suman/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/suman/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/suman/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b


TF-IDF scores 0.0 on this paraphrase (no shared vocabulary); the real embedding model correctly ranks KB005 (the actual returns-policy article) as the top match, because it captures meaning, not just word overlap.

## 3. The full retrieval pipeline

In [5]:
def retrieve(query, top_k=3, min_similarity=0.25):
    query_embedding = model.encode([query])
    scores = cosine_similarity(query_embedding, doc_embeddings)[0]
    kb_scored = kb.copy()
    kb_scored["similarity"] = scores
    top = kb_scored.sort_values("similarity", ascending=False).head(top_k)
    return top[top["similarity"] >= min_similarity]

for query in [
    "How long do I have to send something back?",
    "Do you sell electronics accessories",
]:
    print(f"\nQuery: {query}")
    result = retrieve(query, top_k=2)
    if result.empty:
        print("  No relevant article found")
    else:
        for row in result.itertuples():
            print(f"  {row.similarity:.3f} — {row.doc_id}: {row.title}")


Query: How long do I have to send something back?
  0.652 — KB005: Return Window for Standard Items
  0.523 — KB007: Damaged Item on Arrival

Query: Do you sell electronics accessories
  No relevant article found


/Users/suman/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/suman/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/suman/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
/Users/suman/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/suman/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/suman/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b


## 4. Intro to vector databases

Storing the same embeddings in Chroma, and querying it the same way a production system would — the index handles the similarity search internally.

In [6]:
import chromadb

# Pass our own precomputed sentence-transformer embeddings directly, rather than letting
# Chroma download and use its own default embedding model.
chroma_client = chromadb.Client()
collection = chroma_client.get_or_create_collection("support_kb")

collection.add(
    documents=kb["content"].tolist(),
    ids=kb["doc_id"].tolist(),
    embeddings=doc_embeddings.tolist(),
    metadatas=[{"title": t} for t in kb["title"].tolist()]
)

query_vec = model.encode(["How long do I have to send something back?"]).tolist()
chroma_results = collection.query(query_embeddings=query_vec, n_results=3)
for doc_id, meta, distance in zip(chroma_results["ids"][0], chroma_results["metadatas"][0], chroma_results["distances"][0]):
    print(f"{doc_id}: {meta['title']} (distance={distance:.3f})")

KB005: Return Window for Standard Items (distance=0.697)
KB007: Damaged Item on Arrival (distance=0.953)
KB006: Refund Processing Time (distance=1.027)


**Note:** Chroma reports *distance* (lower = more similar), the inverse framing of the *similarity* scores (higher = more similar) used above — same underlying computation, opposite direction of the number line. Either convention is fine as long as your code is consistent about which one it's using.

## 5. Try it yourself

Query both `retrieve(...)` and the Chroma `collection.query(...)` with a question about the loyalty program or store hours, and confirm they agree on the top result even though one reports similarity and the other reports distance.

In [7]:
my_query = "What are the store timings in Gurugram?"

print("retrieve():")
print(retrieve(my_query, top_k=1)[["doc_id", "title", "similarity"]])

print("\nChroma:")
my_query_vec = model.encode([my_query]).tolist()
chroma_result = collection.query(query_embeddings=my_query_vec, n_results=1)
print(chroma_result["ids"][0], chroma_result["metadatas"][0])

retrieve():
   doc_id                    title  similarity
12  KB013  Store Hours in Gurugram    0.658297

Chroma:
['KB013'] [{'title': 'Store Hours in Gurugram'}]


/Users/suman/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/suman/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/suman/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
